<a href="https://colab.research.google.com/github/iamdreams1/Group_051_-COMP90042_Project_2026/blob/MoritzLaurer%2FDeBERTa-v3-large-mnli-fever-anli-ling-wanli%E6%A8%A1%E5%9E%8B%E6%80%A7%E8%83%BD%E6%B5%8B%E8%AF%95/Group_051__COMP90042_Project_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
# 1.1 Path and device configuration (team workflow: Drive shortcut on Colab, relative path locally)
#
# Design:
#   - Colab : ROOT = /content/drive/MyDrive/comp90042
#             (the team has set up a shortcut so every member sees this folder under MyDrive)
#   - Local : ROOT = absolute path of the directory containing the notebook
#             (after cloning the repo, data/ and cache/ live next to the notebook; no code changes needed)
#   - Any user can override via the COMP90042_ROOT env var (useful when moving disks)
import os, sys, torch

IN_COLAB = 'google.colab' in sys.modules

# 1) Environment variable takes precedence (highest priority; users moving to SSD/external disk just export it)
ROOT = os.environ.get('COMP90042_ROOT')

# 2) Colab: after mounting Drive, use the team's shortcut path
if ROOT is None and IN_COLAB:
    from google.colab import drive
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/comp90042'

# 3) Local: relative to the current working directory (start jupyter from the notebook folder)
if ROOT is None:
    ROOT = os.path.abspath(os.getcwd())

DATA  = os.path.join(ROOT, 'data')
CACHE = os.path.join(ROOT, 'cache')
CKPT  = os.path.join(ROOT, 'ckpt')
for p in (CACHE, CKPT):
    os.makedirs(p, exist_ok=True)

# Device priority: cuda > mps > cpu
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'IN_COLAB : {IN_COLAB}')
print(f'ROOT     : {ROOT}')
print(f'DEVICE   : {DEVICE}')
print(f'DATA     : {DATA}')
print(f'CACHE    : {CACHE}')
print(f'CKPT     : {CKPT}')

In [ ]:
# 1.1.b Data-availability sanity check (team-shared caches: avoid re-running
# heavy steps for every contributor).
#
# This cell only checks whether the expected files are present; loading and
# validation happen later via the cache helpers.
import os

REQUIRED = {
    'data/train-claims.json'           : True,
    'data/dev-claims.json'             : True,
    'data/test-claims-unlabelled.json' : True,
    'data/evidence.json'               : True,
}

OFFICIAL_CACHE = {
    'cache/evidence.pkl'          : 'raw evidence pickle',
    'cache/evidence_filtered.pkl' : 'filtered evidence dict + ids',
    'cache/bm25.pkl'              : 'BM25Okapi index',
    'cache/bge_evidence_emb.npy'  : 'raw BGE evidence embeddings (A1.a ablation)',
    'cache/bge_faiss.index'       : 'raw BGE FAISS index (A1.a ablation)',
    'cache/retrieval_eval_dev.pkl': 'A1.a retrieval Recall@K table',
    'cache/bm25_dev.pkl'          : 'bm25 dev candidates (baseline / Exp-1)',
    'cache/bm25_train.pkl'        : 'bm25 train candidates (baseline / Exp-1)',
    'cache/bm25_test.pkl'         : 'bm25 test candidates (baseline / Exp-1)',
    # Custom BGE retriever cache (built by A2.b on first run, ~3.5 GB)
    'cache/custom_bge_evidence_emb.npy': 'fine-tuned BGE evidence embeddings',
    'cache/custom_bge_faiss.index'    : 'fine-tuned BGE FAISS index',
    'cache/custom_bge_evi_ids.pkl'    : 'fine-tuned BGE evidence id order',
}

print(f'Checking ROOT = {ROOT}')
print('-' * 70)
missing_required = []
for rel, required in REQUIRED.items():
    p = os.path.join(ROOT, rel)
    if os.path.exists(p):
        print(f'  ok {rel:<42s} {os.path.getsize(p)/1e6:>8.1f} MB')
    else:
        print(f'  MISSING {rel}')
        if required:
            missing_required.append(rel)

if missing_required:
    raise FileNotFoundError(
        f'Missing required files: {missing_required}\n'
        f'Make sure the Drive shortcut is mounted, or place data/ next to '
        f'{ROOT} when running locally.'
    )
print('-' * 70)
print('Required data is in place')

print('\nQuick existence check of official caches (presence only — '
      'validation happens later)')
print('-' * 70)
missing_cache = []
for rel, desc in OFFICIAL_CACHE.items():
    p = os.path.join(ROOT, rel)
    if os.path.exists(p):
        print(f'  ok {rel:<34s} {os.path.getsize(p)/1e6:>8.1f} MB  ({desc})')
    else:
        print(f'  missing {rel:<34s} ({desc})')
        missing_cache.append(rel)

# Fast-path summary: cache readiness per experiment route
BM25_CACHE_RELS = ['cache/bm25_dev.pkl', 'cache/bm25_train.pkl', 'cache/bm25_test.pkl']
CUSTOM_BGE_RELS = ['cache/custom_bge_evidence_emb.npy', 'cache/custom_bge_faiss.index', 'cache/custom_bge_evi_ids.pkl']
ALL_BM25_FILES_PRESENT       = all(os.path.exists(os.path.join(ROOT, rel)) for rel in BM25_CACHE_RELS)
ALL_CUSTOM_BGE_FILES_PRESENT = all(os.path.exists(os.path.join(ROOT, rel)) for rel in CUSTOM_BGE_RELS)
print('\nFast-path summary:')
print(f'  Exp-1 baseline (BM25) cache present       : {ALL_BM25_FILES_PRESENT}')
print(f'  Exp-2/3 custom BGE corpus cache present   : {ALL_CUSTOM_BGE_FILES_PRESENT}')
print('                   (if False, cell A2.b rebuilds on first run, ~30 minutes)')

In [ ]:
# 1.2 NLTK resources (uncomment the pip line below on a fresh Colab session)
!pip install -q rank_bm25 sentencepiece protobuf tiktoken sentence-transformers==2.7.0 transformers==4.40.0 nltk pandas matplotlib scikit-learn peft==0.9.0 accelerate

import importlib.util
import nltk

for module_name in ['sentencepiece', 'transformers', 'tokenizers']:
    if importlib.util.find_spec(module_name) is None:
        raise RuntimeError(f'{module_name} failed to install; rerun this cell, and if needed Runtime > Restart runtime followed by Run all')

def ensure_nltk_resource(pkg, path, required=True):
    try:
        nltk.data.find(path)
    except (LookupError, OSError):
        ok = nltk.download(pkg, quiet=True)
        if required and not ok:
            raise RuntimeError(f'Failed to download NLTK resource {pkg}')

ensure_nltk_resource('punkt',      'tokenizers/punkt')
ensure_nltk_resource('punkt_tab',  'tokenizers/punkt_tab', required=False)  # only available on newer nltk versions
ensure_nltk_resource('stopwords',  'corpora/stopwords')
print('Tokenizer dependencies ready (sentencepiece / transformers / tokenizers)')
print('NLTK resources ready')

In [ ]:
# 1.2.b Cache helpers (must be defined before any heavy cache is consumed)
import os, pickle, gc, time

# Experiment routes (three lines):
#   Exp-1 baseline : BM25              -> DeBERTa-v3-base
#   Exp-2 large    : custom BGE+rerank -> DeBERTa-v3-large + LoRA
#   Exp-3 NLI      : custom BGE+rerank -> MoritzLaurer NLI + LoRA    <-- final choice
# We removed the Hybrid BM25-OR-BGE route to avoid duplicating custom_end_to_end.

FORCE_REBUILD = {
    'filtered': False,
    'bm25': False,
    'bge': False,
    'custom_bge': False,
    'eval': False,
}

CACHE_META = {
    'filtered': {
        'version': 1,
        'max_words': 256,
        'min_words_threshold': 3,
    },
    'bm25': {
        'version': 1,
        'tokenizer': 'nltk_word_tokenize_lower_stopwords_punct_isalnum_v1',
        'max_words': 256,
        'min_words_threshold': 3,
    },
    'bge': {
        # raw BGE — used for the A1.a retrieval ablation (vs. the fine-tuned version)
        'version': 1,
        'model': 'BAAI/bge-base-en-v1.5',
        'max_seq_len': 384,
        'embedding_dim': 768,
    },
    'custom_bge': {
        # Fine-tuned BGE retriever (Drive: ckpt/custom-bge-retriever-final)
        # + fine-tuned reranker (Drive: ckpt/custom-bge-reranker-final).
        # Used by the main route (Exp-2 / Exp-3).
        'version': 1,
        'base_model': 'BAAI/bge-base-en-v1.5',
        'retriever_ckpt': 'ckpt/custom-bge-retriever-final',
        'reranker_ckpt': 'ckpt/custom-bge-reranker-final',
        'max_seq_len': 384,
        'embedding_dim': 768,
    },
}


def save_pickle_atomic(data, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)


def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


def release_large_objects(*names):
    released = []
    for name in names:
        if name in globals():
            del globals()[name]
            released.append(name)
    if released:
        gc.collect()
        if 'torch' in globals() and getattr(torch, 'cuda', None) and torch.cuda.is_available():
            torch.cuda.empty_cache()
        print('released:', ', '.join(released))


def validate_candidate_cache(data, claims, name='cache', max_len=None, known_evidence=None):
    if not isinstance(data, dict):
        return False, f'{name} is not a dict'
    expected = set(claims.keys())
    actual = set(data.keys())
    missing = expected - actual
    extra = actual - expected
    if missing or extra:
        return False, f'{name} claim ids mismatch: missing={len(missing)} extra={len(extra)}'
    for cid, vals in data.items():
        if not isinstance(vals, list):
            return False, f'{name}[{cid}] is not a list'
        if max_len is not None and len(vals) > max_len:
            return False, f'{name}[{cid}] has {len(vals)} candidates > {max_len}'
        if len(vals) != len(set(vals)):
            return False, f'{name}[{cid}] contains duplicate evidence ids'
        if known_evidence is not None:
            bad = [eid for eid in vals if eid not in known_evidence]
            if bad:
                return False, f'{name}[{cid}] contains unknown evidence id {bad[0]}'
    return True, 'ok'


def load_candidate_cache(path, claims, name='cache', max_len=None, known_evidence=None, force=False):
    if force or not os.path.exists(path):
        reason = 'forced rebuild' if force else 'missing file'
        print(f'  {name}: {reason}')
        return None
    try:
        t0 = time.time()
        data = load_pickle(path)
        ok, msg = validate_candidate_cache(data, claims, name=name, max_len=max_len, known_evidence=known_evidence)
        if ok:
            avg = sum(len(v) for v in data.values()) / max(1, len(data))
            print(f'  loaded {path} ({len(data)} claims, avg {avg:.0f}, {time.time()-t0:.1f}s)')
            return data
        print(f'  invalid {name}: {msg}; rebuilding')
    except Exception as e:
        print(f'  failed to load {name} ({type(e).__name__}: {e}); rebuilding')
    return None

print('Cache-first helpers ready')
print('FORCE_REBUILD =', FORCE_REBUILD)

In [ ]:
# 1.3 Load data
import os, json, pickle, time

os.makedirs(CACHE, exist_ok=True)

def load_json(name):
    with open(f'{DATA}/{name}.json') as f:
        return json.load(f)

t0 = time.time()
train    = load_json('train-claims')
dev      = load_json('dev-claims')
test     = load_json('test-claims-unlabelled')
baseline = load_json('dev-claims-baseline')
print(f'Claims loaded in {time.time()-t0:.1f}s | '
      f'train={len(train)} dev={len(dev)} test={len(test)} baseline={len(baseline)}')

# evidence.json is 174MB; we pickle it on first load and only read the pickle afterwards.
evi_pkl = f'{CACHE}/evidence.pkl'
if os.path.exists(evi_pkl):
    print('Loading evidence from the pickle cache ...')
    t0 = time.time()
    evidence = load_pickle(evi_pkl)
    print(f'  loaded in {time.time()-t0:.1f}s | {len(evidence):,} passages')
else:
    print('First-time load of evidence.json (174MB) ...')
    t0 = time.time()
    evidence = load_json('evidence')
    print(f'  loaded in {time.time()-t0:.1f}s | {len(evidence):,} passages')
    save_pickle_atomic(evidence, evi_pkl)
    print(f'  cached -> {evi_pkl}')

evi_ids = list(evidence.keys())
print(f'\nFirst 3 evidence ids: {evi_ids[:3]}')
print(f'Evidence corpus size: {len(evi_ids):,} passages')

In [ ]:
# 1.4 Sample one training claim and inspect its schema
import random
random.seed(0)

cid = random.choice(list(train.keys()))
sample = train[cid]
print(f'claim_id     : {cid}')
print(f'claim_text   : {sample["claim_text"]}')
print(f'claim_label  : {sample["claim_label"]}')
print(f'gold evi ids : {sample["evidences"]}')
print()
for eid in sample['evidences'][:2]:
    print(f'  [{eid}] {evidence[eid][:200]} ...')

# Test split comparison (no label, no gold evidences)
print()
print('Test fields (should not contain claim_label / evidences):')
print(f'  {list(next(iter(test.values())).keys())}')

## 1.5 EDA: length distribution, label distribution, evidence count

Four sub-plots in one figure for direct reuse in the report:

| Sub-plot | Purpose |
|---|---|
| Claim length | Informs tokenizer max_length on the claim side |
| 4-way label distribution | Motivates class weighting; DISPUTED is very rare |
| Gold evidence count per claim | Informs the final output top-k size |
| Evidence length (log scale) | Informs the evidence-side truncation strategy |

In [ ]:
# 1.5 EDA: 4-panel figure
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

LABELS = ['SUPPORTS', 'REFUTES', 'NOT_ENOUGH_INFO', 'DISPUTED']

# Train-claim view
df = pd.DataFrame([{
    'claim_len' : len(v['claim_text'].split()),
    'label'     : v['claim_label'],
    'n_evi'     : len(v['evidences']),
} for v in train.values()])

# Evidence length (computed once; used by the panels below)
evi_lens = np.array([len(evidence[eid].split()) for eid in evi_ids])

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# (1) Claim length
df['claim_len'].hist(ax=axes[0,0], bins=30, color='steelblue', edgecolor='white')
axes[0,0].set_title('Train: claim length (words)')
axes[0,0].set_xlabel('words'); axes[0,0].set_ylabel('count')
axes[0,0].axvline(df['claim_len'].mean(), color='red', linestyle='--',
                  label=f"mean={df['claim_len'].mean():.1f}")
axes[0,0].legend()

# (2) 4-way label distribution
label_counts = df['label'].value_counts().reindex(LABELS)
label_counts.plot(kind='bar', ax=axes[0,1],
                  color=['#2ecc71','#e74c3c','#95a5a6','#f39c12'])
axes[0,1].set_title('Train: 4-class label distribution')
axes[0,1].set_xticklabels(LABELS, rotation=15)
for i, v in enumerate(label_counts):
    axes[0,1].text(i, v + 5, str(v), ha='center')

# (3) Gold evidence count per claim
n_evi_max = df['n_evi'].max()
df['n_evi'].hist(ax=axes[1,0], bins=range(1, n_evi_max + 2),
                 color='darkorange', edgecolor='white')
axes[1,0].set_title('Train: # gold evidence per claim')
axes[1,0].set_xlabel('# evidence'); axes[1,0].set_ylabel('count')
axes[1,0].axvline(df['n_evi'].mean(), color='red', linestyle='--',
                  label=f"mean={df['n_evi'].mean():.2f}")
axes[1,0].legend()

# (4) Evidence length (log y because of the heavy tail)
axes[1,1].hist(evi_lens, bins=80, color='purple', edgecolor='white')
axes[1,1].set_yscale('log')
axes[1,1].set_title('Evidence corpus: passage length (words)')
axes[1,1].set_xlabel('words'); axes[1,1].set_ylabel('count (log)')
axes[1,1].axvline(256, color='red', linestyle='--', label='256w cutoff')
axes[1,1].legend()

plt.tight_layout()
plt.savefig(f'{CACHE}/eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()

# Print headline numbers for the report
print(f'Claim length: mean={df["claim_len"].mean():.1f}, p95={df["claim_len"].quantile(0.95):.0f}')
print(f'Gold evidence count: mean={df["n_evi"].mean():.2f}, p95={df["n_evi"].quantile(0.95):.0f}')
print(f'Evidence length: p50={int(np.percentile(evi_lens,50))}, '
      f'p95={int(np.percentile(evi_lens,95))}, p99={int(np.percentile(evi_lens,99))}, '
      f'fraction >256 words: {(evi_lens > 256).mean()*100:.2f}%')

In [ ]:
# 1.6 Evidence-filtering cache (loads cache/evidence_filtered.pkl by default)
import time, gc
from collections import Counter

MAX_WORDS = CACHE_META['filtered']['max_words']
MIN_WORDS_THRESHOLD = CACHE_META['filtered']['min_words_threshold']
FILTERED_CACHE = f'{CACHE}/evidence_filtered.pkl'


def validate_filtered_cache(obj):
    if not isinstance(obj, dict):
        return False, 'cache is not a dict'
    required = {'meta', 'evidence_filtered', 'evi_ids_filtered', 'evidence_tags', 'original_count'}
    if not required.issubset(obj):
        return False, f'missing keys: {sorted(required - set(obj))}'
    meta = obj.get('meta', {})
    if meta.get('max_words') != MAX_WORDS or meta.get('min_words_threshold') != MIN_WORDS_THRESHOLD:
        return False, 'filter config mismatch'
    ef = obj['evidence_filtered']
    ids = obj['evi_ids_filtered']
    tags = obj['evidence_tags']
    if not isinstance(ef, dict) or not isinstance(ids, list) or not isinstance(tags, dict):
        return False, 'invalid object types'
    if len(ef) != len(ids) or ids != list(ef.keys()):
        return False, 'filtered evidence id order mismatch'
    return True, 'ok'

loaded_filtered = False
if os.path.exists(FILTERED_CACHE) and not FORCE_REBUILD['filtered']:
    try:
        t0 = time.time()
        obj = load_pickle(FILTERED_CACHE)
        ok, msg = validate_filtered_cache(obj)
        if ok:
            evidence_filtered = obj['evidence_filtered']
            evi_ids_filtered = obj['evi_ids_filtered']
            evidence_tags = obj['evidence_tags']
            original_evidence_count = obj['original_count']
            loaded_filtered = True
            print(f'Loaded filtered evidence cache in {time.time()-t0:.1f}s -> {FILTERED_CACHE}')
        else:
            print(f'Invalid filtered cache: {msg}; rebuilding')
    except Exception as e:
        print(f'Failed to load filtered cache ({type(e).__name__}: {e}); rebuilding')

if not loaded_filtered:
    if 'evidence' not in globals():
        raise RuntimeError('Raw evidence is missing. Run the data loading cell first.')
    t0 = time.time()
    evidence_filtered = {}
    evidence_tags = {}
    for eid, text in evidence.items():
        word_count = len(text.split())
        if word_count > MAX_WORDS:
            continue
        evidence_filtered[eid] = text
        evidence_tags[eid] = 'short' if word_count < MIN_WORDS_THRESHOLD else 'standard'
    evi_ids_filtered = list(evidence_filtered.keys())
    original_evidence_count = len(evidence)
    obj = {
        'meta': dict(CACHE_META['filtered']),
        'evidence_filtered': evidence_filtered,
        'evi_ids_filtered': evi_ids_filtered,
        'evidence_tags': evidence_tags,
        'original_count': original_evidence_count,
    }
    save_pickle_atomic(obj, FILTERED_CACHE)
    print(f'Built filtered evidence in {time.time()-t0:.1f}s -> {FILTERED_CACHE}')

print(f'Original passages: {original_evidence_count:,}')
print(f'Remaining passages: {len(evidence_filtered):,}')
print(f'Removed (Too Long): {original_evidence_count - len(evidence_filtered):,}')
print(f"Tagged as 'Short':   {list(evidence_tags.values()).count('short'):,}")

# Only the filtered corpus is needed downstream; freeing raw evidence reduces Colab RAM peak.
release_large_objects('evidence', 'evi_ids')

In [ ]:
train_pairs = []

# Process all claims in train-claims.json
for cid, data in train.items():
    label = data['claim_label']
    claim_text = data['claim_text']
    gold_ids = data.get('evidences', [])

    for eid in gold_ids:
        # Check if the evidence exists in our filtered dictionary
        if eid in evidence_filtered:
            train_pairs.append({
                "claim": claim_text,
                "evidence": evidence_filtered[eid],
                "label": label,
                "claim_id": cid,
                "evidence_id": eid
            })

print(f"Total training pairs generated: {len(train_pairs):,}")

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string

nltk.download('punkt')
nltk.download('stopwords')

# Pre-loading stopwords and punctuation for speed
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)

In [ ]:
def nltk_tokenize(text):
    # Standard NLTK tokenization
    tokens = word_tokenize(text.lower())

    # Filter out stopwords and punctuation to keep the index clean
    # and improve retrieval relevance for scientific terms
    filtered = [
        t for t in tokens
        if t not in stop_words and t not in punctuation and t.isalnum()
    ]
    return filtered

In [ ]:
# 2.1 BM25 cache (loads cache/bm25.pkl by default; rebuilds only when missing or invalid)
import time, gc
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

BM25_CACHE = f'{CACHE}/bm25.pkl'
BM25_META = dict(CACHE_META['bm25'])


def unpack_bm25_cache(obj):
    # New format: {'meta': ..., 'bm25': ..., 'evi_ids_filtered': ...}
    if isinstance(obj, dict) and {'bm25', 'evi_ids_filtered'}.issubset(obj):
        return obj.get('bm25'), obj.get('evi_ids_filtered'), obj.get('meta'), False
    # Backward-compatible legacy format already used by this project: (bm25, ids)
    if isinstance(obj, tuple) and len(obj) == 2:
        return obj[0], obj[1], None, True
    return None, None, None, False


def validate_bm25_cache(obj):
    cached_bm25, cached_ids, meta, legacy = unpack_bm25_cache(obj)
    if cached_bm25 is None or cached_ids is None:
        return False, 'unrecognized BM25 cache format', None
    if list(cached_ids) != list(evi_ids_filtered):
        return False, 'evidence id order mismatch', None
    if getattr(cached_bm25, 'corpus_size', None) != len(evi_ids_filtered):
        return False, 'BM25 corpus size mismatch', None
    if meta is not None and meta != BM25_META:
        return False, 'BM25 tokenizer/filter metadata mismatch', None
    if legacy:
        print('  note: legacy BM25 cache has no metadata; accepted after id-order and corpus-size checks')
    return True, 'ok', cached_bm25


def build_bm25_cache():
    print('Tokenizing filtered evidence with NLTK... (only runs on cache miss/force rebuild)')
    t0 = time.time()
    corpus_tokens = []
    for eid in tqdm(evi_ids_filtered, desc='BM25 tokenize'):
        corpus_tokens.append(nltk_tokenize(evidence_filtered[eid]))
    print(f'Tokenization finished in {time.time()-t0:.1f}s')

    print('Building BM25Okapi index...')
    t0 = time.time()
    built_bm25 = BM25Okapi(corpus_tokens)
    del corpus_tokens
    gc.collect()
    print(f'BM25 built in {time.time()-t0:.1f}s')

    save_pickle_atomic({
        'meta': BM25_META,
        'bm25': built_bm25,
        'evi_ids_filtered': list(evi_ids_filtered),
    }, BM25_CACHE)
    print(f'BM25 cache saved -> {BM25_CACHE}')
    return built_bm25

bm25 = None
if os.path.exists(BM25_CACHE) and not FORCE_REBUILD['bm25']:
    try:
        t0 = time.time()
        obj = load_pickle(BM25_CACHE)
        ok, msg, loaded_bm25 = validate_bm25_cache(obj)
        if ok:
            bm25 = loaded_bm25
            print(f'BM25 loaded from cache in {time.time()-t0:.1f}s -> {BM25_CACHE}')
        else:
            print(f'Invalid BM25 cache: {msg}; rebuilding')
    except Exception as e:
        print(f'Failed to load BM25 cache ({type(e).__name__}: {e}); rebuilding')

if bm25 is None:
    bm25 = build_bm25_cache()

In [ ]:
def get_top_200(claim_text, n=200):
    required = ['bm25', 'evi_ids_filtered', 'evidence_filtered', 'nltk_tokenize']
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            'Missing prerequisites: ' + ', '.join(missing) +
            '. Run data filtering, tokenizer setup, and BM25 cache cells first.'
        )

    query_tokens = nltk_tokenize(claim_text)
    scores = bm25.get_scores(query_tokens)

    import numpy as np
    n = max(1, min(n, len(scores)))
    top_indices = np.argsort(scores)[-n:][::-1]

    results = []
    for idx in top_indices:
        eid = evi_ids_filtered[idx]
        results.append({
            'id': eid,
            'text': evidence_filtered[eid],
            'score': float(scores[idx]),
        })
    return results

# Lightweight smoke test: BM25 only, no dense/BGE objects involved.
if 'train' not in globals() or len(train) == 0:
    raise RuntimeError('`train` is missing or empty. Please run data loading cell first.')

example_cid = 'claim-1937' if 'claim-1937' in train else next(iter(train))
example_claim = train[example_cid]['claim_text']
top_evidences = get_top_200(example_claim)
print(f'Example claim ID: {example_cid}')
print(f"Top result ID: {top_evidences[0]['id']}")


In [ ]:
# 2.2 Dense retrieval dependencies (install faiss only when BGE/FAISS is needed)
import importlib.util, subprocess, sys, os

EVAL_CACHE_PATH = f'{CACHE}/retrieval_eval_dev.pkl'

NEED_DENSE_DEPS = (
    FORCE_REBUILD['bge'] or
    FORCE_REBUILD['custom_bge'] or
    FORCE_REBUILD['eval'] or
    (not os.path.exists(EVAL_CACHE_PATH)) or
    (not os.path.exists(f'{CACHE}/custom_bge_evidence_emb.npy'))
)

if NEED_DENSE_DEPS and importlib.util.find_spec('faiss') is None:
    print('Installing faiss-cpu because dense retrieval may be needed...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'])
else:
    print(f'Dense dependency check complete. NEED_DENSE_DEPS={NEED_DENSE_DEPS}')

In [ ]:
# 2.x BGE / FAISS dense retrieval (lazy init; loaded only when the cache is missing or a force rebuild is requested)
import os, gc, time, math
import numpy as np
import torch

CHUNK_SIZE  = 50_000
MAX_SEQ_LEN = CACHE_META['bge']['max_seq_len']
SORT_BY_LEN = True

CHUNK_DIR  = f'{CACHE}/bge_chunks'
EMB_FINAL  = f'{CACHE}/bge_evidence_emb.npy'
ORDER_FILE = f'{CACHE}/bge_sort_order.npy'
INDEX_PATH = f'{CACHE}/bge_faiss.index'
os.makedirs(CHUNK_DIR, exist_ok=True)

DENSE_READY = False
model_bge = None
index = None
embeddings = None


def _dense_device_config():
    if torch.cuda.is_available():
        return 'cuda', True, 512
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return 'mps', False, 128
    return 'cpu', False, 32


def _load_or_create_sort_order(n):
    if SORT_BY_LEN:
        if os.path.exists(ORDER_FILE):
            sort_order = np.load(ORDER_FILE)
            if len(sort_order) != n:
                raise RuntimeError(f'{ORDER_FILE} length mismatch: {len(sort_order)} != {n}')
            print(f'Loaded sort order -> {ORDER_FILE}')
            return sort_order
        print('Computing length sort order...')
        lens = np.fromiter((len(evidence_filtered[eid]) for eid in evi_ids_filtered), dtype=np.int32, count=n)
        sort_order = np.argsort(lens, kind='stable').astype(np.int64)
        np.save(ORDER_FILE, sort_order)
        return sort_order
    return np.arange(n, dtype=np.int64)


def ensure_dense_retrieval(force_rebuild=False):
    """Load/build BGE model + FAISS index only when a later cell truly needs dense retrieval."""
    global DENSE_READY, model_bge, index, embeddings
    if DENSE_READY and not force_rebuild:
        return

    import faiss
    from sentence_transformers import SentenceTransformer

    device, use_fp16, batch_size = _dense_device_config()
    n = len(evi_ids_filtered)
    print(f'Dense retrieval init: device={device} fp16={use_fp16} batch={batch_size} passages={n:,}')

    model_bge = SentenceTransformer(CACHE_META['bge']['model'], device=device)
    model_bge.max_seq_length = MAX_SEQ_LEN
    if use_fp16:
        model_bge = model_bge.half()

    sort_order = _load_or_create_sort_order(n)
    n_chunks = math.ceil(n / CHUNK_SIZE)

    need_rebuild_embeddings = force_rebuild or FORCE_REBUILD['bge'] or not os.path.exists(EMB_FINAL)
    if need_rebuild_embeddings:
        done = sum(os.path.exists(f'{CHUNK_DIR}/chunk_{ci:04d}.npy') for ci in range(n_chunks))
        print(f'Encoding BGE chunks: {done}/{n_chunks} already present')
        overall_t0 = time.time()
        for ci in range(n_chunks):
            chunk_path = f'{CHUNK_DIR}/chunk_{ci:04d}.npy'
            if os.path.exists(chunk_path) and not force_rebuild:
                continue
            start = ci * CHUNK_SIZE
            end = min(start + CHUNK_SIZE, n)
            idxs = sort_order[start:end]
            texts = [evidence_filtered[evi_ids_filtered[i]] for i in idxs]

            t0 = time.time()
            emb = model_bge.encode(
                texts,
                batch_size=batch_size,
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=True,
            )
            if not np.isfinite(emb).all():
                raise RuntimeError(f'chunk {ci} has NaN/Inf; stop before writing corrupted cache')

            emb = emb.astype('float16' if use_fp16 else 'float32')
            tmp = chunk_path + '.tmp.npy'
            np.save(tmp, emb)
            os.replace(tmp, chunk_path)
            del emb, texts
            gc.collect()
            if device == 'cuda':
                torch.cuda.empty_cache()
            dt = time.time() - t0
            eta = dt * (n_chunks - ci - 1) / 60
            print(f'  chunk {ci+1:>3}/{n_chunks} [{end-start} passages] {dt:.1f}s ETA {eta:.1f}min')

        print(f'BGE encoding wall time {(time.time()-overall_t0)/60:.1f}min')
        print('Merging chunks into final embedding file...')
        dim = model_bge.get_sentence_embedding_dimension()
        merged = np.empty((n, dim), dtype='float32')
        for ci in range(n_chunks):
            start = ci * CHUNK_SIZE
            end = min(start + CHUNK_SIZE, n)
            part = np.load(f'{CHUNK_DIR}/chunk_{ci:04d}.npy')
            if part.dtype != np.float32:
                part = part.astype('float32')
            merged[sort_order[start:end]] = part
            del part
            gc.collect()
        tmp = EMB_FINAL + '.tmp.npy'
        np.save(tmp, merged)
        os.replace(tmp, EMB_FINAL)
        embeddings = merged
        print(f'Saved embeddings -> {EMB_FINAL} shape={embeddings.shape}')
    else:
        embeddings = None
        print(f'Embedding cache exists -> {EMB_FINAL}')

    need_rebuild_index = force_rebuild or FORCE_REBUILD['bge'] or not os.path.exists(INDEX_PATH)
    if os.path.exists(INDEX_PATH) and not need_rebuild_index:
        t0 = time.time()
        index = faiss.read_index(INDEX_PATH)
        if index.ntotal != n:
            raise RuntimeError(f'FAISS ntotal mismatch: {index.ntotal} != {n}. Set FORCE_REBUILD["bge"] = True.')
        if index.d != CACHE_META['bge']['embedding_dim']:
            raise RuntimeError(f'FAISS dim mismatch: {index.d} != {CACHE_META["bge"]["embedding_dim"]}. Set FORCE_REBUILD["bge"] = True.')
        print(f'FAISS index loaded -> {INDEX_PATH} ntotal={index.ntotal:,} dim={index.d} {time.time()-t0:.1f}s')
        if embeddings is None:
            embeddings = np.load(EMB_FINAL, mmap_mode='r')
            expected_shape = (n, CACHE_META['bge']['embedding_dim'])
            if embeddings.shape != expected_shape:
                raise RuntimeError(f'Embedding shape mismatch: {embeddings.shape} != {expected_shape}. Set FORCE_REBUILD["bge"] = True.')
            print(f'embeddings mmap shape={embeddings.shape} dtype={embeddings.dtype}')
    else:
        if embeddings is None:
            embeddings = np.load(EMB_FINAL).astype('float32')
            print(f'Loaded embeddings into RAM for FAISS build: shape={embeddings.shape}')
        expected_shape = (n, CACHE_META['bge']['embedding_dim'])
        if embeddings.shape != expected_shape:
            raise RuntimeError(f'Embedding shape mismatch: {embeddings.shape} != {expected_shape}. Set FORCE_REBUILD["bge"] = True.')
        faiss.normalize_L2(embeddings)
        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings)
        tmp = INDEX_PATH + '.tmp'
        faiss.write_index(index, tmp)
        os.replace(tmp, INDEX_PATH)
        print(f'FAISS index built -> {INDEX_PATH} ntotal={index.ntotal:,} dim={embeddings.shape[1]}')

    DENSE_READY = True

if FORCE_REBUILD['bge']:
    ensure_dense_retrieval(force_rebuild=True)
elif not ALL_HYBRID_FILES_PRESENT:
    print('Some hybrid cache files are missing; dense retrieval will be initialized on demand.')
else:
    print('All hybrid cache files are present; BGE/FAISS initialization is skipped until a cache miss requires it.')

## A1: Retrieval ablation — BM25 / Custom BGE / Custom BGE + Reranker (Recall@K)

Three retrieval routes side-by-side, matching the "retrieval upgrade" experiment narrative:

1. **BM25** — used by Exp-1 baseline, feeds DeBERTa-v3-base
2. **Custom BGE** — fine-tuned dense retriever
3. **Custom BGE + Reranker** — final choice, feeds DeBERTa-v3-large / MoritzLaurer NLI

Cell A1.a computes the three Recall@K rows. Cell A1.c caches the BM25 top-200 candidates needed by the baseline experiment.
Custom BGE + Reranker candidates are produced dynamically by cell A2.b (no separate cache: only dev/test = 153×2 claims, fast enough).

In [ ]:
# A1.a: Retrieval evaluation — BM25 vs Custom BGE vs Custom BGE+Reranker (dev = 154 claims)
# Output: a table + cache/retrieval_eval_dev.pkl (idempotent)
import numpy as np
import time
from tqdm.auto import tqdm

EVAL_CACHE_PATH = f'{CACHE}/retrieval_eval_dev.pkl'

def recall_at_k(candidate_dict, claims, k):
    rs = []
    for cid, c in claims.items():
        pred = set(candidate_dict.get(cid, [])[:k])
        gold = set(c['evidences'])
        if not gold:
            continue
        rs.append(len(pred & gold) / len(gold))
    return float(np.mean(rs)) if rs else 0.0

def f_at_k_simple(candidate_dict, claims, k):
    fs = []
    for cid, c in claims.items():
        pred = set(candidate_dict.get(cid, [])[:k])
        gold = set(c['evidences'])
        if not pred or not gold:
            fs.append(0.0)
            continue
        tp = len(pred & gold)
        if tp == 0:
            fs.append(0.0)
            continue
        p, r = tp / len(pred), tp / len(gold)
        fs.append(2 * p * r / (p + r))
    return float(np.mean(fs)) if fs else 0.0


# A1.a builds this single table only; it does NOT cache all-train candidates
# (those are produced by A1.c / A2.b to avoid duplicated storage).
KS = [3, 5, 10, 50, 100]
results = {}

# --- 1) BM25 alone (top-100 covers the entire K range) ---
print('[A1.a] BM25 top-100 ...')
t0 = time.time()
bm25_dev_cand = {cid: [d['id'] for d in get_top_200(c['claim_text'], n=100)]
                 for cid, c in tqdm(dev.items(), desc='BM25')}
print(f'  bm25 done in {time.time()-t0:.1f}s')
results['BM25'] = {k: recall_at_k(bm25_dev_cand, dev, k) for k in KS}

# --- 2) Custom BGE alone (lazy-load the fine-tuned retriever) ---
print('\n[A1.a] Custom BGE top-100 ...')
try:
    from sentence_transformers import SentenceTransformer
    import faiss as _faiss
    custom_retriever_path = f'{CKPT}/custom-bge-retriever-final'
    if not os.path.isdir(custom_retriever_path):
        raise FileNotFoundError(f'{custom_retriever_path} missing — see README "Drive checkpoint sync"')
    _retr = SentenceTransformer(custom_retriever_path, device=DEVICE)
    _emb_path = f'{CACHE}/custom_bge_evidence_emb.npy'
    _idx_path = f'{CACHE}/custom_bge_faiss.index'
    _ids_path = f'{CACHE}/custom_bge_evi_ids.pkl'
    if os.path.exists(_emb_path) and os.path.exists(_idx_path) and os.path.exists(_ids_path):
        _idx = _faiss.read_index(_idx_path)
        _eids = load_pickle(_ids_path)
    else:
        print('  custom-bge corpus cache missing; skip Custom BGE evaluation (run cell A2.b first to build it)')
        raise RuntimeError('skip custom_bge eval')
    bge_dev_cand = {}
    for cid, c in tqdm(dev.items(), desc='Custom BGE'):
        q = _retr.encode([c['claim_text']], normalize_embeddings=True).astype('float32')
        _, ix = _idx.search(q, 100)
        bge_dev_cand[cid] = [_eids[j] for j in ix[0]]
    results['Custom BGE'] = {k: recall_at_k(bge_dev_cand, dev, k) for k in KS}
    del _retr, _idx
except Exception as e:
    print(f'  Custom BGE evaluation skipped ({type(e).__name__}: {e})')

# --- 3) Custom BGE + Reranker (reuses final_dev_evidence from cell A2.b; optional) ---
if 'final_dev_evidence' in globals() and final_dev_evidence is not None and RETRIEVAL_SOURCE == 'custom_end_to_end':
    print('\n[A1.a] Custom BGE + Reranker (reusing final_dev_evidence from A2.b) ...')
    results['Custom BGE + Reranker'] = {k: recall_at_k(final_dev_evidence, dev, k)
                                          for k in KS if k <= FINAL_TOP_K + 5}
else:
    print('\n[A1.a] Reranker row requires cell A2.b to have run with RETRIEVAL_SOURCE=custom_end_to_end')

# --- print table ---
print('\n=== Recall@K (dev) ===')
header = f'{"Retriever":<26s}' + '  '.join(f'R@{k:>3d}' for k in KS)
print(header)
print('-' * len(header))
for name in ['BM25', 'Custom BGE', 'Custom BGE + Reranker']:
    if name not in results:
        continue
    row = f'{name:<26s}'
    for k in KS:
        v = results[name].get(k)
        row += f'  {v:>4.3f}' if v is not None else '  ---- '
    print(row)

# cache results for reuse
save_pickle_atomic({'meta': {'k_grid': KS}, 'results': results}, EVAL_CACHE_PATH)
print(f'\nsaved {EVAL_CACHE_PATH}')

In [ ]:
# A1.c: Cache BM25 single-route top-200 candidates — required by the baseline experiment
import os, time
from tqdm.auto import tqdm


def cache_bm25_topk(claims, name, k=200, force_rebuild=False):
    path = f'{CACHE}/bm25_{name}.pkl'
    cached = load_candidate_cache(path, claims, name=f'bm25_{name}', max_len=k,
                                  known_evidence=evidence_filtered, force=force_rebuild)
    if cached is not None:
        return cached
    t0 = time.time()
    out = {}
    for cid, c in tqdm(claims.items(), desc=f'BM25 {name}'):
        out[cid] = [d['id'] for d in get_top_200(c['claim_text'], n=k)]
    save_pickle_atomic(out, path)
    print(f'  saved {path}  ({len(out)} claims, {time.time()-t0:.1f}s)')
    return out


bm25_train = cache_bm25_topk(train, 'train', k=200)
bm25_dev   = cache_bm25_topk(dev,   'dev',   k=200)
bm25_test  = cache_bm25_topk(test,  'test',  k=200)
print('BM25 candidate caches ready: bm25_train / bm25_dev / bm25_test')

# Fine Tuning Embedding model

In [ ]:
!pip install -q faiss-cpu
!pip install --upgrade sentence-transformers

In [ ]:
import random
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments
)
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import InformationRetrievalEvaluator

# --- 1. BUILD THE TRIPLET DATASET ---
print("Formatting Triplet Dataset for MNRL...")
triplets = []

for cid, c in train.items():
    claim_text = c['claim_text']
    gold_eids = set(c.get('evidences', []))

    # Get hard negatives from your Stage 1 cache
    retrieved_eids = hybrid_train.get(cid, [])
    wrong_eids = [eid for eid in retrieved_eids if eid not in gold_eids]

    if not gold_eids or not wrong_eids:
        continue

    # Create (Anchor, Positive, Negative) triplets
    for gold_eid in gold_eids:
        if gold_eid in evidence_filtered:
            # Pick a random hard negative for this gold evidence
            neg_eid = random.choice(wrong_eids)
            if neg_eid in evidence_filtered:
                triplets.append({
                    "anchor": claim_text,
                    "positive": evidence_filtered[gold_eid],
                    "negative": evidence_filtered[neg_eid]
                })

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_list(triplets)
print(f"Generated {len(train_dataset)} training triplets.")


# --- 2. FORMAT THE INFORMATION RETRIEVAL EVALUATOR ---
print("Formatting Dev Set for IR Evaluator...")
queries = {}
relevant_docs = {}

for cid, c in dev.items():
    if c.get('evidences'):
        queries[cid] = c['claim_text']
        relevant_docs[cid] = set(c['evidences'])

# The corpus is just your entire database of evidence paragraphs
corpus = evidence_filtered

dev_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="fact-check-dev-ir",
    show_progress_bar=True
)


# --- 3. INITIALIZE MODEL & LOSS ---
print("Loading BGE-Base Model...")
model = SentenceTransformer("BAAI/bge-base-en-v1.5", device=DEVICE)

# Using Cached MNRL: Simulates a massive batch size of 256 to create perfect
# "In-Batch Negatives", while safely processing them in chunks of 16 to save VRAM.
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=16)


# --- 4. DEFINE TRAINING ARGUMENTS ---
args = SentenceTransformerTrainingArguments(
    output_dir=f"{CKPT}/custom-bge-retriever",
    num_train_epochs=1,
    per_device_train_batch_size=256,  # This works because of Cached MNRL!
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=0.1,  # v5: pass float to mean ratio
    fp16=True,
    # CRITICAL: MNRL relies on comparing different claims in the same batch.
    # NO_DUPLICATES ensures we don't accidentally put the same claim in a batch twice.
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="epoch",      # <--- Change this from "steps"
    save_strategy="epoch",      # <--- Change this from "steps"
    logging_steps=50,
    eval_steps=200,
    report_to="none"
)


# --- 5. EXECUTE TRAINING ---
print("Configuring Trainer...")
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
    evaluator=dev_evaluator,
)

# Run a baseline evaluation before training
# (Disabled: each baseline pass re-encodes the full ~1.2M evidence corpus, ~26 min on Mac/MPS.
#  Re-enable only when you specifically need a pre-training reference number.)
# print("Running Zero-Shot IR Baseline...")
# dev_evaluator(model)

print("Starting Fine-Tuning...")
trainer.train()

# --- 6. SAVE MODEL ---
final_model_path = f"{CKPT}/custom-bge-retriever-final"
model.save_pretrained(final_model_path)
print(f"Training Complete! Stage 1 model saved to {final_model_path}")

## Using reranker to select top 5


In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.util import mine_hard_negatives
from torch.utils.data import DataLoader

# --- 1. PREP THE DATA FOR THE LIBRARY ---
# mine_hard_negatives expects a Hugging Face Dataset with 'query' and 'passage' columns.
# We will populate it with all your true Claim -> Gold Evidence pairs.
print("Formatting positive pairs...")
formatted_data = []

for cid, c in train.items():
    claim_text = c['claim_text']
    gold_eids = set(c.get('evidences', []))

    if not gold_eids:
        continue

    for eid in gold_eids:
        if eid in evidence_filtered:
            formatted_data.append({
                "query": claim_text,
                "passage": evidence_filtered[eid]
            })

# Convert your python list into a Hugging Face Dataset
hf_train_dataset = Dataset.from_list(formatted_data)
print(f"Created Base Dataset with {len(hf_train_dataset)} positive pairs.")


# --- 2. LOAD THE EMBEDDING MODEL ---
# Use the Stage-1 fine-tuned retriever so the mined hard negatives reflect the
# distribution the reranker will actually see at inference time. Fall back to
# stock BGE only if the fine-tuned checkpoint has not been produced yet.
custom_retriever_path = f"{CKPT}/custom-bge-retriever-final"
if os.path.isdir(custom_retriever_path):
    print(f"Loading FINE-TUNED retriever for hard-negative mining -> {custom_retriever_path}")
    embedding_model = SentenceTransformer(custom_retriever_path, device=DEVICE)
else:
    print(f"WARNING: {custom_retriever_path} not found; falling back to BAAI/bge-base-en-v1.5.")
    print("         Run the Fine Tuning Embedding cell first so reranker training sees the same distribution.")
    embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device=DEVICE)


# --- 3. RUN THE OFFICIAL MINER ---
print("Mining hard negatives (this will take a moment to build the FAISS index)...")
hard_train_dataset = mine_hard_negatives(
    dataset=hf_train_dataset,
    model=embedding_model,
    num_negatives=4,               # 4 hard distractors per positive evidence
    margin=0.1,                    # Standard margin for contrastive separation
    sampling_strategy="top",       # Grab the absolute trickiest distractors
    output_format="labeled-pair",  # REQUIRED FOR BCE LOSS! (Returns Query, Passage, Label)
    use_faiss=True,                # Uses FAISS to prevent RAM overload
)

print(f"Mining complete! Generated {len(hard_train_dataset)} total labeled pairs.")


# --- 4. CONVERT TO DATALOADER FOR CROSS-ENCODER ---
print("Formatting for Cross-Encoder training...")
train_examples = []

# The library returns the data as a dictionary with 'query', 'passage', and 'label'
for row in hard_train_dataset:
    # We wrap them in InputExample objects so the CrossEncoder can read them
    train_examples.append(
        InputExample(texts=[row['query'], row['passage']], label=float(row['label']))
    )

# Create the standard PyTorch DataLoader (Batch size 16 is optimal for 16GB GPUs)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
print("DataLoader ready! You can now pass this directly into model.fit()")


In [ ]:
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator

print("Building Dev Set Evaluator from Hybrid Cache...")
eval_samples = []

for cid, c in dev.items():
    claim_text = c['claim_text']
    gold_eids = set(c.get('evidences', []))

    if not gold_eids:
        continue

    # 1. Get the text for the correct evidence
    positive_texts = [evidence_filtered[eid] for eid in gold_eids if eid in evidence_filtered]

    # 2. Get the hard distractors your BM25/BGE hybrid found for this specific claim
    retrieved_eids = hybrid_dev.get(cid, [])
    wrong_eids = [eid for eid in retrieved_eids if eid not in gold_eids]

    # Take the top 50 negatives for a robust evaluation (just like the documentation)
    negative_texts = [evidence_filtered[eid] for eid in wrong_eids[:50] if eid in evidence_filtered]

    # 3. Only add if we have both positives and negatives to evaluate against
    if positive_texts and negative_texts:
        eval_samples.append({
            "query": claim_text,
            "positive": positive_texts,
            "negative": negative_texts
        })

print(f"Created {len(eval_samples)} evaluation samples.")

# Initialize the Evaluator
dev_evaluator = CrossEncoderRerankingEvaluator(
    samples=eval_samples,
    batch_size=16,
    name="fact-check-dev"
)

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder
# Assuming 'model' is your freshly loaded CrossEncoder("BAAI/bge-reranker-base")
model = CrossEncoder("BAAI/bge-reranker-base", num_labels=1, max_length=384, device=DEVICE)
print("Running Zero-Shot Baseline Evaluation...")
baseline_results = dev_evaluator(model)

print("\n--- ZERO-SHOT BASELINE ---")
for metric, score in baseline_results.items():
    print(f"{metric}: {score:.4f}")

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoderTrainingArguments, CrossEncoderTrainer
from sentence_transformers.evaluation import SequentialEvaluator
from sentence_transformers.cross_encoder.evaluation import CrossEncoderNanoBEIREvaluator

# A. Your Custom Fact-Checking Evaluator (Built in the previous step)
# dev_evaluator = CrossEncoderRerankingEvaluator(samples=eval_samples, name="fact-check-dev")

# B. The NanoBEIR Zero-Shot Evaluator
print("Downloading NanoBEIR datasets...")
nano_beir_evaluator = CrossEncoderNanoBEIREvaluator(
    dataset_names=["msmarco", "nfcorpus", "nq"],
    batch_size=16, # Keep this the same as your training batch size
)

# C. Bundle them together!
seq_evaluator = SequentialEvaluator([dev_evaluator, nano_beir_evaluator])


# --- 2. DEFINE TRAINING ARGUMENTS ---
args = CrossEncoderTrainingArguments(
    output_dir=f"{CKPT}/custom-bge-reranker",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    # CRITICAL: Tell the trainer how often to run the evaluators
    eval_strategy="steps",
    eval_steps=200,          # Runs the SequentialEvaluator every 200 steps
    save_strategy="epoch",
    logging_steps=50,
    report_to="none"
)


# --- 3. INITIALIZE TRAINER & TRAIN ---
print("Configuring Trainer...")
trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=hard_train_dataset,
    evaluator=seq_evaluator  # <--- Pass the bundled evaluators here!
)

print("Starting Fine-Tuning with Real-Time Evaluation...")
trainer.train()

# --- 4. SAVE FINAL MODEL ---
final_model_path = f"{CKPT}/custom-bge-reranker-final"
trainer.save_model(final_model_path)
print(f"Training Complete! Model saved to {final_model_path}")

In [ ]:
import os
from sentence_transformers import CrossEncoder

# (Make sure final_model_path is defined)
final_model_path = f"{CKPT}/custom-bge-reranker-final"

print("========================================")
print("🔍 RUNNING MODEL SAVE VERIFICATION")
print("========================================\n")

# --- STEP 1: File System Check ---
print("--- 1. File System Check ---")
if os.path.exists(final_model_path):
    files = os.listdir(final_model_path)
    print(f"Folder exists! Found {len(files)} files.")

    # We are looking for the two most critical files
    has_config = "config.json" in files
    has_weights = "model.safetensors" in files or "pytorch_model.bin" in files

    if has_config and has_weights:
        print("✅ Core model weights and config files are present.")
    else:
        print("❌ WARNING: Missing critical model files. Google Drive might still be syncing!")
        print(f"Currently in folder: {files}")
else:
    print(f"❌ ERROR: The folder {final_model_path} does not exist at all.")

# --- STEP 2 & 3: Load and Inference Check ---
print("\n--- 2. Load & Inference Test ---")
try:
    # local_files_only=True is CRITICAL. It forces the code to use your saved files
    # instead of accidentally downloading a fresh model from the internet.
    test_model = CrossEncoder(final_model_path, local_files_only=True)
    print("✅ Model successfully loaded into memory from the saved folder!")

    # Run a tiny dummy prediction to ensure the tensor math isn't corrupted
    dummy_pair = [["The capital of France is Paris.", "Paris is the capital and most populous city of France."]]
    score = test_model.predict(dummy_pair)

    print(f"✅ Prediction engine is fully functional! (Dummy score: {score[0]:.4f})")
    print("\n🎉 VERIFICATION PASSED: Your model is safely stored and ready for tomorrow!")

except Exception as e:
    print(f"\n❌ FATAL ERROR: Failed to load or run the model. The save might be corrupted.\nError Details:\n{e}")

In [ ]:
# A2.b Final evidence selection (the retrieval -> classifier interface)
#
# Retrieval route selector (must match the chosen downstream classifier):
#   'bm25'              -> Exp-1 baseline (feeds DeBERTa-base)
#   'custom_end_to_end' -> Exp-2 large / Exp-3 NLI (main route)
#
# Output: final_{train,dev,test}_evidence: Dict[claim_id, List[evidence_id]]
# These dicts are the sole evidence source used by A3 training and A4 inference.

import numpy as np
import faiss
from tqdm.auto import tqdm
from sentence_transformers import CrossEncoder, SentenceTransformer

# ==========================================
# --- 1. MODEL SETUP (only when custom_end_to_end is selected) ---
# ==========================================

RETRIEVAL_SOURCE = 'custom_end_to_end'   # 'bm25' | 'custom_end_to_end'
FINAL_TOP_K = 5
FORCE_REBUILD_CUSTOM_BGE = False         # flip True to invalidate the corpus cache

# All three splits go through custom_end_to_end. Train (~11k claims) two-stage
# retrieval takes 30-60 minutes; for a quick check temporarily set ('dev', 'test').
RUN_SPLITS = ('train', 'dev', 'test')

_SPLIT_DATA = {'train': train, 'dev': dev, 'test': test}
final_train_evidence = None
final_dev_evidence   = None
final_test_evidence  = None


def rerank_evidence(claim, candidate_eids, evidence_dict, top_k=5):
    if not candidate_eids:
        return []
    pairs = [[claim, evidence_dict.get(eid, '')] for eid in candidate_eids]
    scores = rerank_model.predict(pairs)
    sorted_indices = np.argsort(scores)[::-1].tolist()
    return [candidate_eids[i] for i in sorted_indices[:top_k]]


def apply_dynamic_reranking(cache_dict, claims_dict, top_k=5, max_candidates=50, desc='Reranking'):
    reranked = {}
    for cid, c in tqdm(claims_dict.items(), desc=desc):
        cand = cache_dict.get(cid, [])[:max_candidates]
        reranked[cid] = rerank_evidence(c['claim_text'], cand, evidence_filtered, top_k=top_k)
    return reranked


def generate_stage1_cache(claims_dict, model, faiss_index, evidence_ids, top_k=50):
    new_cache = {}
    for cid, c in tqdm(claims_dict.items(), desc='Stage 1 Searching'):
        q = model.encode([c['claim_text']], normalize_embeddings=True)
        q = np.array(q).astype('float32')
        _, ix = faiss_index.search(q, top_k)
        new_cache[cid] = [evidence_ids[j] for j in ix[0]]
    return new_cache


def build_custom_bge_index(retriever, evidence_dict, cache_dir, force_rebuild=False):
    """Encode the full corpus with the fine-tuned retriever; cache to disk.

    Cache files under cache_dir:
      custom_bge_evidence_emb.npy  — float32 (N, dim)
      custom_bge_faiss.index       — FAISS IndexFlatIP
      custom_bge_evi_ids.pkl       — evidence id order (must match emb rows)
    """
    emb_path   = f'{cache_dir}/custom_bge_evidence_emb.npy'
    index_path = f'{cache_dir}/custom_bge_faiss.index'
    ids_path   = f'{cache_dir}/custom_bge_evi_ids.pkl'

    evidence_ids   = list(evidence_dict.keys())
    evidence_texts = list(evidence_dict.values())
    expected_dim   = retriever.get_sentence_embedding_dimension()
    n              = len(evidence_ids)

    if not force_rebuild and os.path.exists(emb_path) and os.path.exists(index_path) and os.path.exists(ids_path):
        try:
            cached_ids = load_pickle(ids_path)
            if list(cached_ids) != evidence_ids:
                print('  custom-bge cache: evidence id order mismatch; rebuilding')
            else:
                emb = np.load(emb_path, mmap_mode='r')
                if emb.shape != (n, expected_dim):
                    print(f'  custom-bge cache: shape mismatch {emb.shape} != {(n, expected_dim)}; rebuilding')
                else:
                    idx = faiss.read_index(index_path)
                    if idx.ntotal != n or idx.d != expected_dim:
                        print(f'  custom-bge cache: FAISS dim/ntotal mismatch (ntotal={idx.ntotal}, d={idx.d}); rebuilding')
                    else:
                        print(f'  custom-bge cache HIT -> {index_path} (ntotal={idx.ntotal:,}, dim={idx.d})')
                        return idx, evidence_ids
        except Exception as e:
            print(f'  custom-bge cache load failed ({type(e).__name__}: {e}); rebuilding')

    print(f'Encoding corpus with fine-tuned retriever ({n:,} passages)...')
    emb = retriever.encode(
        evidence_texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True
    )
    emb = np.ascontiguousarray(emb, dtype='float32')
    if not np.isfinite(emb).all():
        raise RuntimeError('custom-bge embeddings contain NaN/Inf; refusing to write cache')

    idx = faiss.IndexFlatIP(emb.shape[1])
    idx.add(emb)

    os.makedirs(cache_dir, exist_ok=True)
    tmp = emb_path + '.tmp.npy'
    np.save(tmp, emb); os.replace(tmp, emb_path)
    tmp = index_path + '.tmp'
    faiss.write_index(idx, tmp); os.replace(tmp, index_path)
    save_pickle_atomic(evidence_ids, ids_path)
    print(f'  saved -> {emb_path} shape={emb.shape}')
    print(f'  saved -> {index_path} ntotal={idx.ntotal:,}')
    return idx, evidence_ids


# ==========================================
# --- 2. FINAL EVIDENCE SELECTION ---
# ==========================================

if RETRIEVAL_SOURCE == 'custom_end_to_end':
    custom_retriever_path = f'{CKPT}/custom-bge-retriever-final'
    custom_reranker_path  = f'{CKPT}/custom-bge-reranker-final'
    assert os.path.isdir(custom_retriever_path), (
        f'missing {custom_retriever_path}\n'
        f'Please sync custom-bge-retriever-final/ from Colab Drive into ckpt/ (see README)'
    )
    assert os.path.isdir(custom_reranker_path), (
        f'missing {custom_reranker_path}\n'
        f'Please sync custom-bge-reranker-final/ from Colab Drive into ckpt/ (see README)'
    )

    print('Loading Stage 2: FINE-TUNED BGE-Reranker...')
    rerank_model = CrossEncoder(custom_reranker_path, device=DEVICE)
    print('Loading Stage 1: FINE-TUNED BGE-Retriever...')
    retriever_model = SentenceTransformer(custom_retriever_path, device=DEVICE)

    print('\n--- Initializing FAISS database (cache-first) ---')
    custom_faiss_index, evidence_ids = build_custom_bge_index(
        retriever_model, evidence_filtered, CACHE,
        force_rebuild=FORCE_REBUILD_CUSTOM_BGE,
    )

    print(f'\n--- Stage 1+2 active splits: {RUN_SPLITS} ---')
    for split in ('train', 'dev', 'test'):
        if split not in RUN_SPLITS:
            print(f'  skipping split={split!r} (not in RUN_SPLITS)')
            continue
        print(f'\n[{split}] Stage 1: wide-net (top 50)')
        stage1 = generate_stage1_cache(_SPLIT_DATA[split], retriever_model, custom_faiss_index, evidence_ids, top_k=50)
        print(f'[{split}] Stage 2: rerank (top {FINAL_TOP_K})')
        reranked = apply_dynamic_reranking(stage1, _SPLIT_DATA[split], FINAL_TOP_K, max_candidates=50, desc=f'Reranking {split}')
        globals()[f'final_{split}_evidence'] = reranked

elif RETRIEVAL_SOURCE == 'bm25':
    # Baseline path: top-5 from the BM25 cache (train/dev/test all need bm25_*.pkl)
    for split in ('train', 'dev', 'test'):
        cache_path = f'{CACHE}/bm25_{split}.pkl'
        assert os.path.exists(cache_path), f'missing {cache_path} — run cell A1.c first'
        globals()[f'final_{split}_evidence'] = load_pickle(cache_path)

else:
    raise ValueError(
        f'Unknown RETRIEVAL_SOURCE: {RETRIEVAL_SOURCE!r} '
        f'(allowed: "bm25" / "custom_end_to_end")'
    )

FINAL_EVIDENCE_SOURCE = RETRIEVAL_SOURCE
TOP_K = FINAL_TOP_K


# --- 3. Evaluation ---
def f_at_k(candidate_dict, claims, k):
    if candidate_dict is None:
        return None
    fs = []
    for cid, c in claims.items():
        pred = set(candidate_dict.get(cid, [])[:k])
        gold = set(c['evidences'])
        if not pred or not gold:
            fs.append(0.0)
            continue
        tp = len(pred & gold)
        if tp == 0:
            fs.append(0.0)
            continue
        p, r = tp / len(pred), tp / len(gold)
        fs.append(2 * p * r / (p + r))
    return float(np.mean(fs))

print(f'\nRETRIEVAL_SOURCE = {RETRIEVAL_SOURCE}')
print(f'FINAL_TOP_K      = {FINAL_TOP_K}')
print(f'RUN_SPLITS       = {RUN_SPLITS}')
print(f'\n--- F@K on dev (final source = {RETRIEVAL_SOURCE}) ---')
if final_dev_evidence is None:
    print('  dev skipped (not in RUN_SPLITS); no F@K to report')
else:
    print(f'{"K":>3}  {"F@K":>8}')
    print('-' * 14)
    for k in [3, 4, 5, 6]:
        f = f_at_k(final_dev_evidence, dev, k)
        marker = ' <- selected' if k == FINAL_TOP_K else ''
        print(f'{k:>3}  {f:>8.3f}{marker}')

## A3: Three-backbone comparison — base / large / nli + two-stage fine-tuning

**Input format**: `[CLS] claim [SEP] evi_1 [SEP] evi_2 ... [SEP]`, attention-mask weighted mean-pooling, followed by a freshly initialised 4-way classification head (we do not reuse the backbone's original NLI head).

**Three routes** (switched via `CLS_BACKBONE_CHOICE` in cell A3.1):

| Route | Backbone | LoRA | Retrieval | Checkpoint |
|---|---|---|---|---|
| `'base'` | `microsoft/deberta-v3-base` (180M) | no (full fine-tune) | BM25 top-5 | `cls_best_base.pt` |
| `'large'` | `microsoft/deberta-v3-large` (435M) | r=16, α=32 | custom BGE+rerank top-5 | `cls_best_large.pt` |
| `'nli'` (final) | `MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli` (435M) | r=16, α=32 | custom BGE+rerank top-5 | `cls_best_nli.pt` |

The NLI backbone has been pre-fine-tuned on MNLI / FEVER / ANLI / LingNLI / WANLI, giving it the strongest starting point for fact-checking (cell A6 quantifies the gain). LoRA targets `query_proj` and `value_proj`; only the LoRA delta and the new classification head are trainable, which keeps the model within the Colab T4 VRAM budget.

**Two-stage training** (run for every route):
1. **Stage 1 — gold supervision**: train with gold evidence to warm up backbone + head.
2. **Stage 2 — retrieved supervision**: load Stage 1, continue with `final_train_evidence` (whichever retrieval route is active) to close the train-eval distribution gap.

Loss: class-weighted cross-entropy (inverse frequency, sum=4, up-weights DISPUTED) + label smoothing 0.1. Optimiser: AdamW with cosine warm-up. fp16 on CUDA. Best checkpoint is selected by dev accuracy.

In [ ]:
# A3.1: Label mapping + class weights + three-backbone configuration
from collections import Counter
import torch

LABELS = ['SUPPORTS', 'REFUTES', 'NOT_ENOUGH_INFO', 'DISPUTED']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

label_counts = Counter(c['claim_label'] for c in train.values())
print('Train label distribution:')
for l in LABELS:
    n = label_counts.get(l, 0)
    pct = n / len(train) * 100
    print(f'  {l:<18s} {n:>4d}  ({pct:>5.1f}%)')

# Inverse-frequency weights normalised so they sum to num_classes (keeps the loss magnitude stable)
inv = [1.0 / max(label_counts.get(l, 1), 1) for l in LABELS]
s = sum(inv)
CLASS_WEIGHTS = torch.tensor([w / s * len(LABELS) for w in inv], dtype=torch.float)
print(f'\nClass weights (sum=4): {[f"{w:.2f}" for w in CLASS_WEIGHTS.tolist()]}')


# ----------------------------------------------------------------------
# Three-route configuration (change this single line to switch experiments)
# ----------------------------------------------------------------------
# 'base'  -> Exp-1 Baseline       : BM25 + microsoft/deberta-v3-base (full FT)
# 'large' -> Exp-2 Classifier Up  : custom BGE+rerank + microsoft/deberta-v3-large + LoRA
# 'nli'   -> Exp-3 FINAL          : custom BGE+rerank + MoritzLaurer NLI + LoRA
CLS_BACKBONE_CHOICE = 'nli'

BACKBONE_CONFIG = {
    'base': {
        'name':        'microsoft/deberta-v3-base',
        'use_lora':    False,
        'max_length':  512,
        'batch_size':  8,
        'grad_accum':  2,
        'lr':          2e-5,
        'epochs':      3,
        'retrieval':   'bm25',
        'pred_prefix': 'baseline-',
        'ckpt_name':   'cls_best_base.pt',
    },
    'large': {
        'name':        'microsoft/deberta-v3-large',
        'use_lora':    True,
        'max_length':  512,
        'batch_size':  4,
        'grad_accum':  4,
        'lr':          2e-4,
        'epochs':      3,
        'retrieval':   'custom_end_to_end',
        'pred_prefix': 'large-',
        'ckpt_name':   'cls_best_large.pt',
    },
    'nli': {
        'name':        'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli',
        'use_lora':    True,
        'max_length':  512,
        'batch_size':  4,
        'grad_accum':  4,
        'lr':          2e-4,
        'epochs':      3,
        'retrieval':   'custom_end_to_end',
        'pred_prefix': 'nli-',
        'ckpt_name':   'cls_best_nli.pt',
    },
}
assert CLS_BACKBONE_CHOICE in BACKBONE_CONFIG, f'unknown CLS_BACKBONE_CHOICE={CLS_BACKBONE_CHOICE!r}'
CFG = BACKBONE_CONFIG[CLS_BACKBONE_CHOICE]
print(f'\n[A3.1] CLS_BACKBONE_CHOICE = {CLS_BACKBONE_CHOICE!r}')
print(f'        backbone  = {CFG["name"]}')
print(f'        retrieval = {CFG["retrieval"]}')
print(f'        LoRA      = {CFG["use_lora"]}  max_len={CFG["max_length"]}  bs={CFG["batch_size"]}  ga={CFG["grad_accum"]}')

In [ ]:
# A3.2: OOP classes — ClaimDS / BaselineClassifier / Trainer
# All three backbones share the same class hierarchy; per-route differences are
# driven by the CFG dictionary defined in cell A3.1:
#   - base : full fine-tuning, no LoRA
#   - large / nli : LoRA r=16, α=32 (target=query_proj/value_proj)
# In every route the classification head is a freshly initialised nn.Linear(h, 4).
# Even on the MoritzLaurer 3-class NLI backbone, the original NLI head is NOT
# reused — we always train a new 4-class head from scratch.

import gc
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model

CLS_BACKBONE = CFG['name']

class ClaimDS(Dataset):
    """Builds [CLS] claim [SEP] evi_1 [SEP] evi_2 ... [SEP] for the classifier.

    use_gold=True  : early training — evidence comes from c['evidences'] (gold)
    use_gold=False : later training / inference — evidence comes from evi_dict[cid][:top_k]
    """
    def __init__(self, claims, evidence_filtered, evi_dict=None, use_gold=True, top_k=5,
                 tokenizer_name=None, max_length=None):
        self.tok = AutoTokenizer.from_pretrained(tokenizer_name or CLS_BACKBONE, use_fast=False)
        self.max_length = max_length or CFG['max_length']
        self.items = []
        sep = ' [SEP] '
        for cid, c in claims.items():
            if use_gold:
                evis = c.get('evidences', [])
            else:
                evis = (evi_dict.get(cid, []) if evi_dict else [])[:top_k]
            evi_text = sep.join(evidence_filtered.get(e, '') for e in evis) or 'no evidence'
            label = LABEL2ID.get(c.get('claim_label', 'NOT_ENOUGH_INFO'), LABEL2ID['NOT_ENOUGH_INFO'])
            self.items.append((c['claim_text'], evi_text, label, cid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        claim, evi, label, _ = self.items[i]
        enc = self.tok(claim, evi, truncation=True, max_length=self.max_length,
                       padding='max_length', return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(label, dtype=torch.long),
        }


class BaselineClassifier(nn.Module):
    """DeBERTa backbone + single linear classification head; large/nli use LoRA, base does full FT."""
    def __init__(self, backbone=None, n_classes=4, dropout=0.15,
                 use_lora=None, lora_r=16, lora_alpha=32):
        super().__init__()
        backbone = backbone or CLS_BACKBONE
        use_lora = CFG['use_lora'] if use_lora is None else use_lora
        self.encoder = AutoModel.from_pretrained(backbone)

        if use_lora:
            print(f'Applying LoRA PEFT to {backbone}... (r={lora_r}, alpha={lora_alpha})')
            peft_config = LoraConfig(
                r=lora_r,
                lora_alpha=lora_alpha,
                target_modules=['query_proj', 'value_proj'],
                lora_dropout=0.1,
                bias='none',
            )
            self.encoder = get_peft_model(self.encoder, peft_config)
            self.encoder.print_trainable_parameters()
        else:
            print(f'Full fine-tuning {backbone} (LoRA disabled)')

        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.cls = nn.Linear(h, n_classes)   # fresh 4-class head (does NOT reuse the backbone's NLI head)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1)   # mean pooling
        return self.cls(self.dropout(pooled))


class Trainer:
    """fp16 (CUDA only) + class-weighted CE + label smoothing + cosine LR + best-ckpt by dev_acc."""
    def __init__(self, model, train_ds, dev_ds, class_weights,
                 device='cuda', lr=None, epochs=None, batch_size=None, grad_accum=None,
                 ckpt_path=None, label_smoothing=0.1, max_grad_norm=1.0):
        from tqdm.auto import tqdm
        self.tqdm = tqdm
        self.device = device
        self.model = model.to(device)
        batch_size = batch_size or CFG['batch_size']
        grad_accum = grad_accum or CFG['grad_accum']
        lr = lr or CFG['lr']
        epochs = epochs or CFG['epochs']

        self.train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        self.dev_loader   = DataLoader(dev_ds,   batch_size=batch_size, shuffle=False)

        # The optimiser only tracks parameters with requires_grad=True
        # (LoRA + cls head; or all parameters when LoRA is disabled).
        self.optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

        steps = max(1, len(self.train_loader) // grad_accum * epochs)
        warmup = max(1, steps // 10)
        self.scheduler = get_cosine_schedule_with_warmup(self.optimizer, warmup, steps)
        self.criterion = nn.CrossEntropyLoss(
            weight=class_weights.to(device), label_smoothing=label_smoothing,
        )
        self.use_amp = (device == 'cuda')
        self.scaler = torch.cuda.amp.GradScaler(enabled=self.use_amp)
        self.epochs = epochs
        self.grad_accum = grad_accum
        self.max_grad_norm = max_grad_norm
        self.ckpt_path = ckpt_path

    def _forward_loss(self, batch):
        ids  = batch['input_ids'].to(self.device)
        mask = batch['attention_mask'].to(self.device)
        y    = batch['label'].to(self.device)
        if self.use_amp:
            with torch.cuda.amp.autocast():
                logits = self.model(ids, mask)
                loss = self.criterion(logits, y)
        else:
            logits = self.model(ids, mask)
            loss = self.criterion(logits, y)
        return loss

    def train(self):
        best = 0.0
        for ep in range(self.epochs):
            self.model.train()
            total = 0.0
            self.optimizer.zero_grad()
            for step, batch in enumerate(self.tqdm(self.train_loader, desc=f'ep{ep}')):
                loss = self._forward_loss(batch) / self.grad_accum
                if self.use_amp:
                    self.scaler.scale(loss).backward()
                else:
                    loss.backward()
                if (step + 1) % self.grad_accum == 0:
                    if self.use_amp:
                        self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)
                    if self.use_amp:
                        self.scaler.step(self.optimizer)
                        self.scaler.update()
                    else:
                        self.optimizer.step()
                    self.optimizer.zero_grad()
                    self.scheduler.step()
                total += loss.item() * self.grad_accum
            avg = total / max(1, len(self.train_loader))
            acc = self.evaluate()
            print(f'  epoch {ep}: loss={avg:.3f}  dev_acc={acc:.3f}')
            if acc > best and self.ckpt_path:
                best = acc
                torch.save(self.model.state_dict(), self.ckpt_path)
                print(f'    saved -> {self.ckpt_path}')
        return best

    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        ok = total = 0
        for batch in self.dev_loader:
            ids  = batch['input_ids'].to(self.device)
            mask = batch['attention_mask'].to(self.device)
            y    = batch['label'].to(self.device)
            pred = self.model(ids, mask).argmax(-1)
            ok    += (pred == y).sum().item()
            total += y.size(0)
        return ok / max(1, total)

In [ ]:
# A3 training entry — picks backbone / ckpt / training data source from CFG.
# Before training we re-check that RETRIEVAL_SOURCE matches CFG['retrieval'], to
# avoid e.g. running the baseline route on top of custom retrieval evidence.

CLS_BEST_PATH  = f'{CKPT}/{CFG["ckpt_name"]}'
FINAL_CKPT_PATH = CLS_BEST_PATH
CLS_FINAL_PATH = FINAL_CKPT_PATH

# Free retrieval-only large objects before classifier training (each route runs once)
release_large_objects(
    'bm25', 'evidence', 'bge_index_obj', 'bge_embeddings', 'bge_corpus_emb',
)

# Consistency check: the retrieval source used for training must match the one
# declared in BACKBONE_CONFIG for this choice.
assert RETRIEVAL_SOURCE == CFG['retrieval'], (
    f"RETRIEVAL_SOURCE={RETRIEVAL_SOURCE!r} does not match BACKBONE_CONFIG[{CLS_BACKBONE_CHOICE!r}]"
    f"['retrieval']={CFG['retrieval']!r} — go back to cell A2.b, fix RETRIEVAL_SOURCE, and rerun"
)
assert final_train_evidence is not None, (
    'final_train_evidence is None — go back to cell A2.b and include train in RUN_SPLITS'
)
assert final_dev_evidence is not None, 'final_dev_evidence is None — same as above'

print(f'[A3 training] choice={CLS_BACKBONE_CHOICE!r}  ckpt={CLS_BEST_PATH}')
print(f'              backbone={CFG["name"]}')
print(f'              train evidence source = {RETRIEVAL_SOURCE}')

# Two-stage training:
#   Stage 1 (gold supervision)      — evidence from c['evidences']; trains head + encoder
#   Stage 2 (retrieved supervision) — evidence from final_*_evidence; adapts to retrieval noise
RUN_STAGE_2 = True   # set False to keep only Stage 1 (faster, slightly lower dev_acc)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE = {DEVICE}')

# Stage 1: gold-supervised
print('\n=== Stage 1: gold-supervised training ===')
train_ds = ClaimDS(train, evidence_filtered, use_gold=True)
dev_ds   = ClaimDS(dev,   evidence_filtered, evi_dict=final_dev_evidence, use_gold=False, top_k=FINAL_TOP_K)
model    = BaselineClassifier()
trainer  = Trainer(model, train_ds, dev_ds, CLASS_WEIGHTS, device=DEVICE, ckpt_path=CLS_BEST_PATH)
best_s1  = trainer.train()
print(f'Stage 1 best dev_acc = {best_s1:.4f}')

if RUN_STAGE_2:
    print('\n=== Stage 2: retrieved-supervised fine-tuning ===')
    model.load_state_dict(torch.load(CLS_BEST_PATH, map_location=DEVICE))
    train_ds2 = ClaimDS(train, evidence_filtered, evi_dict=final_train_evidence, use_gold=False, top_k=FINAL_TOP_K)
    trainer2  = Trainer(model, train_ds2, dev_ds, CLASS_WEIGHTS, device=DEVICE,
                        ckpt_path=CLS_BEST_PATH, epochs=max(1, CFG['epochs'] // 2))
    best_s2 = trainer2.train()
    print(f'Stage 2 best dev_acc = {best_s2:.4f}')

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## A4: End-to-end prediction + three-way comparison

After training in A3, the classifier is paired with retrieval Top-K to produce dev / test predictions.

**Three comparison routes** (toggled by `CLS_BACKBONE_CHOICE` in cell A3.1, then Run All):

| Route | Retrieval | Classifier | Output |
|---|---|---|---|
| Exp-1 baseline | BM25 top-5 | DeBERTa-v3-base | `baseline-{dev,test}-predictions.json` |
| Exp-2 large | custom BGE+rerank top-5 | DeBERTa-v3-large + LoRA | `large-{dev,test}-predictions.json` |
| **Exp-3 NLI (final)** | custom BGE+rerank top-5 | MoritzLaurer NLI + LoRA | `nli-{dev,test}-predictions.json` <br>+ A4.3 per-evi: `nli-per-evi-{dev,test}-predictions.json` <br>+ A4.5 hybrid routing: `hybrid-{dev,test}-predictions.json` |

A4.1 is concatenated-evidence inference (one forward pass per claim, sees 5 evidence passages at once) and runs for every route.
A4.3 / A4.5 only activate on the NLI route (per-evidence NLI + routing are tied to the NLI pre-training).
Cell A6 reads the three prediction sets and renders the comparison table.

In [ ]:
# A4.1: End-to-end prediction + save dev/test predictions
# Output filenames are driven by CFG["pred_prefix"]: baseline- / large- / nli-
import json, torch
from transformers import AutoTokenizer
from tqdm.auto import tqdm

# Load the checkpoint that corresponds to the current CFG
model.load_state_dict(torch.load(FINAL_CKPT_PATH, map_location=DEVICE))
model.eval()


def predict(claims, evidence_candidates, evidence_filtered, k=None,
            tokenizer_name=None, max_length=None, device=DEVICE):
    if k is None:
        k = FINAL_TOP_K
    tokenizer_name = tokenizer_name or CLS_BACKBONE
    max_length = max_length or CFG['max_length']
    tok = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=False)
    sep = ' [SEP] '
    out = {}
    for cid, c in tqdm(claims.items(), desc='predict'):
        evis = list(evidence_candidates.get(cid, []))[:k]
        if not evis:
            evis = list(evidence_candidates.get(cid, []))[:1]
        if not evis:
            evis = [next(iter(evidence_filtered))]   # eval.py requires at least 1 evidence id
        evi_text = sep.join(evidence_filtered.get(eid, '') for eid in evis) or 'no evidence'
        enc = tok(c['claim_text'], evi_text, truncation=True, max_length=max_length,
                  padding='max_length', return_tensors='pt')
        with torch.no_grad():
            ids  = enc['input_ids'].to(device)
            mask = enc['attention_mask'].to(device)
            if device == 'cuda':
                with torch.cuda.amp.autocast():
                    pred = model(ids, mask).argmax(-1).item()
            else:
                pred = model(ids, mask).argmax(-1).item()
        out[cid] = {
            'claim_text': c['claim_text'],
            'claim_label': ID2LABEL[pred],
            'evidences': evis[:6],
        }
    return out


PREFIX = CFG['pred_prefix']   # 'baseline-' | 'large-' | 'nli-'
dev_pred_path  = f'{ROOT}/{PREFIX}dev-predictions.json'
test_pred_path = f'{ROOT}/{PREFIX}test-claims-predictions.json'

print(f'Predicting dev with retrieval={RETRIEVAL_SOURCE} Top-{FINAL_TOP_K} from ckpt {FINAL_CKPT_PATH}')
dev_pred = predict(dev, final_dev_evidence, evidence_filtered, k=FINAL_TOP_K)
with open(dev_pred_path, 'w') as f:
    json.dump(dev_pred, f, indent=2)
print(f'\nSaved {dev_pred_path}  ({len(dev_pred)} claims)')

test_pred = predict(test, final_test_evidence, evidence_filtered, k=FINAL_TOP_K)
with open(test_pred_path, 'w') as f:
    json.dump(test_pred, f, indent=2)
print(f'Saved {test_pred_path}  ({len(test_pred)} claims)')

sample_cid = next(iter(dev_pred))
print(f'\nSample [{sample_cid}]:')
print(f"  claim    : {dev_pred[sample_cid]['claim_text'][:80]}...")
print(f"  pred     : {dev_pred[sample_cid]['claim_label']}")
print(f"  evidence : {dev_pred[sample_cid]['evidences'][:3]}...")

In [ ]:
# A4.2: Invoke the official eval.py to compute Hmean = 2FA / (F + A)
import subprocess

cmd = [
    'python', f'{DATA}/eval.py',
    '--predictions', dev_pred_path,
    '--groundtruth', f'{DATA}/dev-claims.json',
]
print(' '.join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else r.stderr)

## A4.3: NLI-style per-evidence inference (only active on the `'nli'` route)

We re-use the NLI-pre-trained backbone as a per-evidence NLI model, as a counterpart to the concatenated-evidence inference in A4.1:

- For each evidence i we run `(claim, evi_i)` independently → 4-class softmax `p_i`
- Default prediction is `argmax(mean_i p_i)`
- Strength: stronger on SUPPORTS / REFUTES than the concatenated inference
- Weakness: when scoring a single evidence the model almost never emits NEI / DISPUTED, so overall accuracy is below A4.1

> We initially planned a "one evidence strongly supports + another strongly refutes -> DISPUTED" τ-rule.
> A τ sweep on dev shows that no threshold can reliably trigger it (per-evidence probabilities skew SUP/REF),
> so we leave τ=1.01 (rule disabled). DISPUTED is better fixed on the training side (over-sampling / focal loss).

The `base` and `large` routes skip this cell automatically. The output goes to `nli-per-evi-*-predictions.json`,
which is fused with A4.1's `nli-*-predictions.json` by **A4.5 hybrid routing** to produce the final prediction.

In [ ]:
# A4.3: per-evidence NLI inference + aggregation (NLI route only; base/large are skipped)
if CLS_BACKBONE_CHOICE != 'nli':
    print(f'[A4.3] skip per-evidence NLI inference (CHOICE={CLS_BACKBONE_CHOICE!r}, '
          f'only meaningful for the nli backbone)')
else:
    import json, numpy as np, torch
    from collections import Counter
    from transformers import AutoTokenizer
    from tqdm.auto import tqdm

    # Make sure we use the final NLI checkpoint
    model.load_state_dict(torch.load(FINAL_CKPT_PATH, map_location=DEVICE))
    model.eval()

    _NLI_TOK = AutoTokenizer.from_pretrained(CLS_BACKBONE, use_fast=False)
    SUP = LABEL2ID['SUPPORTS']
    REF = LABEL2ID['REFUTES']
    NEI = LABEL2ID['NOT_ENOUGH_INFO']
    DIS = LABEL2ID['DISPUTED']

    NLI_MAX_LEN = 320  # one evidence at a time — shorter than concatenated, saves VRAM

    @torch.no_grad()
    def _probs_for_pair(claim_text, evi_text, max_length=NLI_MAX_LEN):
        enc = _NLI_TOK(claim_text, evi_text, truncation=True, max_length=max_length,
                       padding='max_length', return_tensors='pt')
        ids  = enc['input_ids'].to(DEVICE)
        mask = enc['attention_mask'].to(DEVICE)
        if DEVICE == 'cuda':
            with torch.cuda.amp.autocast():
                logits = model(ids, mask)
        else:
            logits = model(ids, mask)
        return torch.softmax(logits.float(), dim=-1).squeeze(0).cpu().numpy()

    def _aggregate(per_evi_probs, tau_disp):
        """list[np.ndarray(4)] -> (label_id, mean_probs[4])"""
        if not per_evi_probs:
            v = np.zeros(4); v[NEI] = 1.0
            return NEI, v
        arr = np.stack(per_evi_probs, axis=0)
        mean_p = arr.mean(axis=0)
        max_sup = float(arr[:, SUP].max())
        max_ref = float(arr[:, REF].max())
        if max_sup >= tau_disp and max_ref >= tau_disp:
            return DIS, mean_p
        return int(mean_p.argmax()), mean_p

    def _per_evi_cache(claims, evi_dict, k):
        cache = {}
        for cid, c in tqdm(claims.items(), desc='nli-cache'):
            evis = list(evi_dict.get(cid, []))[:k]
            if not evis:
                evis = [next(iter(evidence_filtered))]
            probs = []
            for eid in evis:
                evi_text = evidence_filtered.get(eid, '') or 'no evidence'
                probs.append(_probs_for_pair(c['claim_text'], evi_text))
            cache[cid] = (evis, probs)
        return cache

    print('Caching per-evidence probs on dev (one forward per evidence)...')
    dev_cache = _per_evi_cache(dev, final_dev_evidence, k=FINAL_TOP_K)

    print('\n=== Sweep DISPUTED conflict threshold τ on dev ===')
    def _eval_tau(tau):
        correct = 0
        cm = Counter()
        for cid, (_, probs) in dev_cache.items():
            gold = dev[cid]['claim_label']
            pid, _m = _aggregate(probs, tau_disp=tau)
            plabel = ID2LABEL[pid]
            cm[(gold, plabel)] += 1
            if plabel == gold:
                correct += 1
        return correct / len(dev_cache), cm

    best_tau, best_acc, best_cm = 1.01, -1.0, None
    TAU_GRID = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70, 1.01]
    for tau in TAU_GRID:
        acc, cm = _eval_tau(tau)
        marker = ''
        if acc > best_acc:
            best_acc, best_tau, best_cm = acc, tau, cm
            marker = ' *'
        print(f'  tau={tau:>4.2f}  dev_acc={acc:.4f}{marker}')
    print(f'\nBest tau = {best_tau}  dev_acc = {best_acc:.4f}')

    # Output filenames use a "nli-per-evi-" prefix so they don't collide with A4.1
    nli_dev_pred_path  = f'{ROOT}/nli-per-evi-dev-predictions.json'
    nli_test_pred_path = f'{ROOT}/nli-per-evi-test-claims-predictions.json'

    dev_nli_pred = {}
    for cid, (evis, probs) in dev_cache.items():
        pid, _m = _aggregate(probs, tau_disp=best_tau)
        dev_nli_pred[cid] = {
            'claim_text': dev[cid]['claim_text'],
            'claim_label': ID2LABEL[pid],
            'evidences': evis[:6],
        }
    with open(nli_dev_pred_path, 'w') as f:
        json.dump(dev_nli_pred, f, indent=2)
    print(f'Saved {nli_dev_pred_path}  ({len(dev_nli_pred)} claims)')

    print('\nPredicting test with NLI aggregation...')
    test_cache = _per_evi_cache(test, final_test_evidence, k=FINAL_TOP_K)
    test_nli_pred = {}
    for cid, (evis, probs) in test_cache.items():
        pid, _m = _aggregate(probs, tau_disp=best_tau)
        test_nli_pred[cid] = {
            'claim_text': test[cid]['claim_text'],
            'claim_label': ID2LABEL[pid],
            'evidences': evis[:6],
        }
    with open(nli_test_pred_path, 'w') as f:
        json.dump(test_nli_pred, f, indent=2)
    print(f'Saved {nli_test_pred_path}  ({len(test_nli_pred)} claims)')

    print('\n=== Dev confusion matrix (rows=gold, cols=pred) @ best tau ===')
    header = ' ' * 18 + '  '.join(f'{l[:5]:>5s}' for l in LABELS)
    print(header)
    for gl in LABELS:
        row = [best_cm.get((gl, pl), 0) for pl in LABELS]
        print(f'  {gl:<16s}' + '  '.join(f'{v:>5d}' for v in row))

    NLI_BEST_TAU = best_tau

## A4.5: Hybrid routing — the final scheme on the NLI route

A4.1 concatenated NLI and A4.3 per-evidence NLI are **complementary** across labels: A4.1 is conservative and reliable on NEI, while A4.3 is sharper on SUP/REF. Based on the dev confusion matrices (printed by this cell when run), we route as follows:

```
if concat_pred (A4.1) == 'NOT_ENOUGH_INFO':
    final = NEI            # trust A4.1's NEI judgement
else:
    final = per_evi_pred   # use A4.3 between SUPPORTS / REFUTES
```

The `base` and `large` routes skip this cell automatically. On the NLI route it produces `hybrid-{dev,test}-predictions.json` — **our final submission file**.

> Historical numbers (single run, indicative only — refresh from cell A6 after re-running):
> | Path | A | F | Hmean |
> |---|---|---|---|
> | A4.1 concat (NLI) | 0.5260 | 0.1057 | 0.1761 |
> | A4.3 NLI aggregated | 0.4675 | 0.1057 | 0.1725 |
> | **A4.5 Hybrid** | **0.5325** | **0.1057** | **0.1764** |

In [ ]:
# A4.5: Hybrid routing — baseline-NEI OR NLI-(SUP/REF)  (NLI route only)
if CLS_BACKBONE_CHOICE != 'nli':
    print(f'[A4.5] skip hybrid routing (CHOICE={CLS_BACKBONE_CHOICE!r}, requires nli + A4.3 outputs)')
else:
    import json, subprocess
    from collections import Counter

    def _score(pred_path, gold_dict):
        pred = json.load(open(pred_path))
        cm = Counter()
        correct = total = 0
        for cid, c in gold_dict.items():
            if cid not in pred:
                continue
            gold = c['claim_label']
            p = pred[cid]['claim_label']
            cm[(gold, p)] += 1
            if p == gold:
                correct += 1
            total += 1
        return correct / max(1, total), cm, total

    def _print_cm(cm, title):
        print(f'\n{title}')
        header = ' ' * 18 + '  '.join(f'{l[:5]:>5s}' for l in LABELS)
        print(header)
        for gl in LABELS:
            row = [cm.get((gl, pl), 0) for pl in LABELS]
            print(f'  {gl:<16s}' + '  '.join(f'{v:>5d}' for v in row))

    def _route(base, nli, claims):
        out = {}
        for cid, c in claims.items():
            bp = base.get(cid)
            np_ = nli.get(cid)
            if bp and bp['claim_label'] == 'NOT_ENOUGH_INFO':
                chosen, evis = bp['claim_label'], bp['evidences']
            elif np_ is not None:
                chosen, evis = np_['claim_label'], np_['evidences']
            elif bp is not None:
                chosen, evis = bp['claim_label'], bp['evidences']
            else:
                chosen, evis = 'NOT_ENOUGH_INFO', []
            out[cid] = {'claim_text': c['claim_text'], 'claim_label': chosen, 'evidences': evis}
        return out

    # Read predictions from A4.1 (nli-*-predictions.json) and A4.3 (nli-per-evi-*-predictions.json)
    base_dev = json.load(open(f'{ROOT}/nli-dev-predictions.json'))
    nli_dev  = json.load(open(f'{ROOT}/nli-per-evi-dev-predictions.json'))
    hybrid_dev = _route(base_dev, nli_dev, dev)

    hybrid_dev_path  = f'{ROOT}/hybrid-dev-predictions.json'
    hybrid_test_path = f'{ROOT}/hybrid-test-claims-predictions.json'

    with open(hybrid_dev_path, 'w') as f:
        json.dump(hybrid_dev, f, indent=2)
    print(f'Saved {hybrid_dev_path}  ({len(hybrid_dev)} claims)')

    base_test = json.load(open(f'{ROOT}/nli-test-claims-predictions.json'))
    nli_test  = json.load(open(f'{ROOT}/nli-per-evi-test-claims-predictions.json'))
    hybrid_test = _route(base_test, nli_test, test)
    with open(hybrid_test_path, 'w') as f:
        json.dump(hybrid_test, f, indent=2)
    print(f'Saved {hybrid_test_path}  ({len(hybrid_test)} claims)')

    # Compare three NLI-side paths on dev
    acc_b, cm_b, _ = _score(f'{ROOT}/nli-dev-predictions.json',         dev)
    acc_n, cm_n, _ = _score(f'{ROOT}/nli-per-evi-dev-predictions.json', dev)
    acc_h, cm_h, _ = _score(hybrid_dev_path,                            dev)
    print(f'\nNLI cat (A4.1)     A = {acc_b:.4f}')
    print(f'NLI per-evi (A4.3) A = {acc_n:.4f}   delta vs cat = {acc_n-acc_b:+.4f}')
    print(f'Hybrid (A4.5)      A = {acc_h:.4f}   delta vs cat = {acc_h-acc_b:+.4f}')

    _print_cm(cm_h, '=== Hybrid confusion matrix ===')

    print('\n--- eval.py: hybrid-dev ---')
    r = subprocess.run(['python', f'{DATA}/eval.py',
                        '--predictions', hybrid_dev_path,
                        '--groundtruth', f'{DATA}/dev-claims.json'],
                       capture_output=True, text=True)
    print(r.stdout if r.returncode == 0 else r.stderr)

## A5: End-to-end smoke test

Runs the full retrieve -> rerank -> classify -> eval pipeline on 8 dev claims in under 2 minutes, to confirm nothing is broken.
Prerequisites for this cell: cell A2.b has completed (so `retriever_model` / `rerank_model` / `custom_faiss_index` are in memory) and cell A4.1's `predict()` function is defined.

In [ ]:
# A5: end-to-end smoke test — 8 dev claims through the full pipeline
import os, json, subprocess

REQUIRED_PATHS = {
    'data':       [f'{DATA}/train-claims.json', f'{DATA}/dev-claims.json',
                   f'{DATA}/test-claims-unlabelled.json', f'{DATA}/evidence.json',
                   f'{DATA}/eval.py'],
    'cache':      [f'{CACHE}/evidence_filtered.pkl', f'{CACHE}/bm25.pkl'],
    'checkpoint': [FINAL_CKPT_PATH,
                   f'{CKPT}/custom-bge-retriever-final',
                   f'{CKPT}/custom-bge-reranker-final'],
}

def _check(group, paths):
    missing = [p for p in paths if not os.path.exists(p)]
    print(f'  {group:11s}: {"OK" if not missing else f"MISSING: {missing}"}')
    return not missing

print('=== A5.1 file existence check ===')
all_ok = all(_check(g, ps) for g, ps in REQUIRED_PATHS.items())
if not all_ok:
    print('\nKey files are missing — see the README section "Drive checkpoint sync" '
          'before re-running')
else:
    print('\n=== A5.2 8 dev claims through the full pipeline ===')
    dev_mini = dict(list(dev.items())[:8])

    # Retrieval: stage-1 (50) + stage-2 (5) on the fly; no hybrid_*.pkl dependency
    mini_stage1 = generate_stage1_cache(dev_mini, retriever_model, custom_faiss_index, evidence_ids, top_k=50)
    mini_evi    = apply_dynamic_reranking(mini_stage1, dev_mini, top_k=FINAL_TOP_K, max_candidates=50, desc='A5 mini-rerank')

    # Classification (A4.1 concatenated)
    mini_pred = predict(dev_mini, mini_evi, evidence_filtered, k=FINAL_TOP_K)
    mini_path = f'{ROOT}/a5_smoke_dev.json'
    with open(mini_path, 'w') as f:
        json.dump(mini_pred, f, indent=2)
    print(f'  saved {mini_path}  ({len(mini_pred)} claims)')

    r = subprocess.run(['python', f'{DATA}/eval.py',
                        '--predictions', mini_path,
                        '--groundtruth', f'{DATA}/dev-claims.json'],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(r.stdout)
        print('=== A5 PASS ===')
    else:
        print(f'  eval.py FAILED: {r.stderr}')
        print('=== A5 FAIL ===')

## A6: Three-way comparison table

Once you have run with `CLS_BACKBONE_CHOICE` set to `'base'`, `'large'`, and `'nli'` in turn, this cell reads the three dev prediction JSON files, calls eval.py, and renders a comparison table. A row with `–` means that experiment has not been run yet.

In [ ]:
# A6: read baseline-/large-/nli- predictions and produce the comparison table
import os, json, subprocess
from collections import OrderedDict

EXPERIMENTS = OrderedDict([
    ('Exp-1 Baseline (BM25 + DeBERTa-base)',          'baseline-dev-predictions.json'),
    ('Exp-2 + Retrieval upgrade + DeBERTa-large',     'large-dev-predictions.json'),
    ('Exp-3 Final: NLI backbone (A4.1 cat)',          'nli-dev-predictions.json'),
    ('Exp-3 Final: NLI per-evidence (A4.3)',          'nli-per-evi-dev-predictions.json'),
    ('Exp-3 Final: Hybrid routing (A4.5) [adopted]',  'hybrid-dev-predictions.json'),
])

def _parse_eval(stdout):
    parsed = {}
    for line in stdout.splitlines():
        for key, tag in [('Evidence Retrieval F-score', 'F'),
                         ('Claim Classification Accuracy', 'A'),
                         ('Harmonic Mean', 'Hmean')]:
            if key in line and '=' in line:
                try:
                    parsed[tag] = float(line.rsplit('=', 1)[-1].strip())
                except ValueError:
                    pass
    return parsed if {'F', 'A', 'Hmean'} <= parsed.keys() else None


def _eval(pred_path):
    if not os.path.exists(pred_path):
        return None
    r = subprocess.run(['python', f'{DATA}/eval.py',
                        '--predictions', pred_path,
                        '--groundtruth', f'{DATA}/dev-claims.json'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        return None
    return _parse_eval(r.stdout)


print('| Experiment | F | A | Hmean |')
print('|---|---|---|---|')
for name, fname in EXPERIMENTS.items():
    res = _eval(f'{ROOT}/{fname}')
    if res is None:
        print(f'| {name} | – | – | – |')
    else:
        print(f'| {name} | {res["F"]:.3f} | {res["A"]:.3f} | {res["Hmean"]:.4f} |')
print('\n(Empty rows: set CLS_BACKBONE_CHOICE in cell A3.1 to the missing value and Run All.)')

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*